# 03 · Embudo de transferencia y conectividad fluvial (PLAN §4)

Reto del Grupo 5 — Río Dulce y Livingston, Izabal.

Este notebook documenta el análisis del reto con el código que genera las tablas
del dashboard. Cuatro preguntas:

1. **¿Se puede medir el embudo?** La base oficial registra visitantes y
   tránsito en las fronteras marítimas de Izabal, pero **nunca en el mismo mes**.
2. **¿Por dónde entra y hacia dónde va el visitante?** El cruce 1:1 entre la
   encuesta y la migración.
3. **¿Cómo se llega a cada nodo del corredor?** Means de transporte y la
   cadena de nodos fluvial.
4. **¿Qué se le recomienda a INGUAT?** Cada recomendación se construye a
   partir de los datos de este notebook.

> **Reglas metodológicas aplicadas:** la ausencia de dato no es cero (regla 1);
> la encuesta es muestral y nunca se extrapola a totales (regla 6); los nodos y
> subdestinos no se suman para obtener un total (regla 4).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from src.etl.clean import clean_migration, clean_survey
from src.etl.load import load_migration, load_survey
from src.etl import transform as t
from src.analysis import embudo as E
from src.config import ISABAL_FEEDER_BORDERS, SURVEY_BORDER_MAP

pd.set_option("display.width", 160)

sur = clean_survey(load_survey())
mig = clean_migration(load_migration())
print("encuesta:", sur.shape, "| migración:", mig.shape)

## A. El hallazgo de fondo: 0 meses de solape

Las fronteras marítimas de Izabal (Puerto Barrios, Livingston, Santo Tomás) tienen
**dos series que no se tocan**:

| Tipo de flujo | Registros | Qué es |
|---|---|---|
| Visitantes (`Turista` + `Excursionista`) | 2009-01 → 2016-12 | Persona que pernocta o visita el día |
| Tránsito (`Viajero`) | 2023-01 → 2026-06 | Cruza la frontera sin calificar como visitante |

Sin un mes común, **no existe una tasa de transferencia oficial para el corredor
marítimo**: el numerador y el denominador nunca son del mismo periodo.

In [ ]:
perfil = t.maritime_flow_profile(mig)
display(perfil[["Frontera", "es_destino_maritimo", "es_alimentadora", "meses_visitantes",
                 "meses_transito", "meses_solape", "embudo_medible"]].round(0))

maritimas = perfil[perfil["es_destino_maritimo"]]
print("\nSOLAPE TOTAL en fronteras marítimas:", int(maritimas["meses_solape"].sum()))
print("Fronteras marítimas con embudo medible:", int(maritimas["embudo_medible"].sum()))
print("Fronteras alimentadoras con embudo medible:",
      int(perfil[perfil["es_alimentadora"]]["embudo_medible"].sum()), "de",
      int(perfil["es_alimentadora"].sum()))

# Comprobación explícita del solape mes a mes
for f in ["05 Puerto Barrios", "06 Livingston", "04 Santo Tomás de Castilla"]:
    s = mig[mig["Frontera"].eq(f)]
    vis = set(t._mes(s.loc[s["es_visita"], "fecha"]))
    tra = set(t._mes(s.loc[s["Tipo de Viajero"].eq("Viajero"), "fecha"]))
    print(f"{f:32s} visitantes={len(vis):3d} meses | tránsito={len(tra):3d} meses | solape={len(vis & tra)}")

## B. El mapeo 1:1 frontera ↔ migración

Las 9 fronteras que registran visitantes con destino Izabal/Caribe en la encuesta
tienen todas equivalente exacto en la base oficial (quitando el prefijo numérico).
Eso permite cruzar *dónde entró* (muestral) con *cuántos entraron* (absoluto).

In [ ]:
print("Mapeo (encuesta -> migración):")
for k, v in SURVEY_BORDER_MAP.items():
    marca = "  <- alimentadora de Izabal" if v in ISABAL_FEEDER_BORDERS else ""
    print(f"  {k:30s} -> {v}{marca}")

print("\nAlimentadoras de Izabal:", len(ISABAL_FEEDER_BORDERS))
print("NO alimentan Izabal:", sorted(set(SURVEY_BORDER_MAP.values()) - set(ISABAL_FEEDER_BORDERS)))

# Verificación contra la base oficial
oficiales = set(mig["Frontera"].unique())
faltan = [v for v in SURVEY_BORDER_MAP.values() if v not in oficiales]
print("\nFracciones del mapeo ausentes en la base oficial:", faltan or "ninguna")

## C. Tasa de transferencia por frontera

Proporción de los **visitantes muestreados** en cada frontera que declara destino
Izabal/Caribe. Es una proporción de la muestra (regla 6): **nunca** se multiplica
por la migración para obtener visitantes de Izabal.

El intervalo de Wilson al 95% acota la incertidumbre por tamaño de muestra.

In [ ]:
tr = t.survey_transfer_rate(sur, mig)
todos = tr[tr["anio"].eq("Todos")].copy()
cols = ["frontera_nombre", "registros_total", "registros_izabal", "tasa_transferencia",
        "ic_inf", "ic_sup", "peso_en_izabal", "migracion_visitantes", "migracion_transito"]
display(todos[cols].sort_values("peso_en_izabal", ascending=False).round(4))

top2 = todos.nlargest(2, "peso_en_izabal")
cuota = top2["peso_en_izabal"].sum()
print(f"\nTop 2 fronteras: {', '.join(top2['frontera_nombre'])}")
print(f"Concentran el {cuota:.1%} de la muestra de visitantes con destino Izabal.")

## D. La cadena de conectividad: frontera → nodo

La tabla que reconstruye el corredor: de dónde entra el visitante y hacia qué nodo
se dirige. Los nodos **no se suman** (regla 4): cada fila se reporta con su
participación dentro de su frontera.

In [ ]:
flujo = t.survey_border_node_flow(sur)
matriz = (flujo.pivot_table(index="frontera_nombre", columns="nodo",
                             values="registros", aggfunc="sum", fill_value=0))
display(matriz)

fig = go.Figure(go.Heatmap(
    z=matriz.to_numpy(), x=list(matriz.columns), y=list(matriz.index),
    colorscale=[[0, "#F2F5F6"], [0.5, "#3FA8B8"], [1, "#0E7C86"]],
    hovertemplate="%{y} → %{x}<br>%{z:,.0f} registros<extra></extra>",
    colorbar=dict(title="reg.", thickness=12, outlinewidth=0)))
fig.update_layout(title="Cadena de conectividad: frontera de entrada → nodo del corredor",
                  height=430, margin=dict(l=10, r=10, t=45, b=10))
fig.update_xaxes(tickangle=-30)
fig.update_yaxes(autorange="reversed")
fig.show()

print("\nNodos de agua y su peso en la muestra:")
display(E.participacion_nodos_agua(flujo).query("es_nodo_agua").round(4))

## E. ¿Con qué se llega a cada nodo?

**Limitación estructural:** la taxonomía de la encuesta (11 categorías) **no
incluye transporte acuático**. La conectividad fluvial, entonces, no se mide como
modo de transporte sino como cadena espacial de nodos. Lo que sí es observable es
el medio tierra/aire con el que se llega al nodo.

In [ ]:
nt = t.survey_node_transport(sur)
print("Taxonomía completa de tipo_transporte en la encuesta:")
print(sur["tipo_transporte"].value_counts().to_string())
print("\n¿Existe alguna categoría acuática?",
      [x for x in sur["tipo_transporte"].unique()
       if any(k in str(x).lower() for k in ["lancha", "barco", "embarc", "maria", "puerto"]) ] or "NO")

print("\nModo dominante por nodo de agua:")
display(E.modo_por_nodo(nt, ["Río Dulce", "Puerto Barrios", "Livingston"]).round(4))

## F. Los cuellos de botella del corredor

Dos medidas: la **estacionalidad del tránsito** (2023–jun 2026) y la
**participación de Izabal en el corredor** de visitantes (2009–2016, la única
ventana con las 5 fronteras comparables).

In [ ]:
tr_mar = mig[mig["Frontera"].isin(["05 Puerto Barrios", "06 Livingston",
                                     "04 Santo Tomás de Castilla"])
             & mig["Tipo de Viajero"].eq("Viajero")]
serie = tr_mar.groupby("fecha")["Viajero"].sum().sort_index()
media = serie.mean()
cv = serie.std() / media
print(f"Tránsito 2023-2026: total {serie.sum():,.0f} | media mensual {media:,.0f} | CV {cv:.3f}")
print("Índice estacional por mes:")
idx = (serie / media).groupby(serie.index.month).mean()
for m, v in idx.items():
    print(f"  {pd.Timestamp(2024, m, 1):%B}: {v:.3f}")
print(f"\nMínimo en {serie.idxmin():%B} ({serie.min():,.0f}) | Máximo en {serie.idxmax():%B} ({serie.max():,.0f})")

cs = t.corridor_share(mig)
cs["fecha"] = pd.to_datetime(cs["fecha"])
print("\nParticipación del corredor (corregida: la ausencia de dato es NaN, no 0):")
display(cs[cs["fecha"].between("2015-09-01", "2017-03-01")].round(4))
print("Filas con 0 visitantes de Izabal (deben ser 0):",
      int((cs["viajeros_izabal"] == 0).sum()))
print("Filas con NaN (ausencia de dato):", int(cs["viajeros_izabal"].isna().sum()))

## G. Itinerarios multi-parada: un supuesto documentado

El campo `otros_destinos` registra itinerarios de dos o más paradas con códigos
(320, 340, 222, 484, 998) que **no traen diccionario**. Se reporta la estructura
de co-ocurrencia —que es un hecho— pero **no se les asigna nombre**: varias
lecturas son consistentes con los datos, así que nombrarlos sería inventar.
Pedir el catálogo es una de las recomendaciones a INGUAT.

In [ ]:
it = E.perfil_itinerarios(sur)
display(it.head(12).round(4))
multi = it[it["es_multi_parada"]]
print(f"\nRegistros con 2 o más paradas: {multi['registros'].sum():,} "
      f"({multi['registros'].sum() / it['registros'].sum():.1%} del campo informado)")
print("Combinaciones más frecuentes:", ", ".join(multi.head(4)["combo"].tolist()))
print("\nSUPUESTO: los códigos pertenecen a un catálogo por base (V10/V11/V12) que no")
print("viene con diccionario. No se nombran; se pide el catálogo a INGUAT.")

## H. Recomendaciones a INGUAT

Cada recomendación se genera desde los datos de este notebook: el texto y su
evidencia no pueden divergir.

In [ ]:
recs = E.recomendaciones_inguat({
    "perfil": perfil,
    "flujo": E.cadena_flujo(flujo),
    "transporte": nt,
    "transito": tr_mar,
    "itinerarios": it,
})
for r in recs:
    print(f"[{r['prioridad']}] {r['titulo']}")
    print(f"   HALLAZGO: {r['hallazgo']}")
    print(f"   ACCIÓN:   {r['recomendacion']}\n")

## Salidas procesadas

`python -m src.etl.pipeline` escribe en `data/processed/`:

| Archivo | Contenido |
|---|---|
| `maritime_flow_profile.parquet` | Cobertura y solape por frontera (el hallazgo central) |
| `survey_border_node_flow.parquet` | Cadena frontera → nodo del corredor |
| `survey_node_transport.parquet` | Modo de transporte por nodo |
| `survey_itinerarios.parquet` | Co-ocurrencia de códigos de destino |
| `survey_transfer_rate.parquet` | Tasa de transferencia con IC de Wilson |

In [ ]:
from src.data import load

for n in ["maritime_flow_profile", "survey_border_node_flow", "survey_node_transport",
          "survey_itinerarios", "survey_transfer_rate"]:
    print(f"{n:32s} {load(n).shape}")